# 15 — Experiment Runner with Logging

**Maps to:** `report/Chapters/Task3.tex` §`T3:Setup`.
**Ticket:** TICKET-15.

Notebook driver that runs N independent GA trials per configuration, logs
best fitness, mean fitness, and diversity per generation to CSV, and supports
multiprocessing and resumability.

In [1]:
from __future__ import annotations

import hashlib
import json
import multiprocessing as mp
import time
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import tsplib95

---
## Scaffolded Dependencies

All operator functions below are copied from their respective ticket
notebooks (or from the scaffolds in notebook 10). Replace each block
with `%run` imports once the upstream notebook is finalised.

In [2]:
# ── SCAFFOLD: Benchmark Loader (TICKET-04) ──────────────────────

def load_tsp(path):
    problem = tsplib95.load(str(path))
    nodes = list(problem.get_nodes())
    coords = np.array([problem.node_coords[n] for n in nodes], dtype=np.float64)
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    return coords, np.sqrt((diff ** 2).sum(axis=-1))


# ── SCAFFOLD: Fitness (TICKET-05) ───────────────────────────────

def tour_length(tour, dist_matrix):
    return dist_matrix[tour, np.roll(tour, -1)].sum()


def is_valid_permutation(chromosome, n_cities):
    return len(chromosome) == n_cities and set(chromosome) == set(range(n_cities))


# ── SCAFFOLD: Population Init (TICKET-06) ───────────────────────

def random_population(pop_size, n_cities, rng):
    return np.array([rng.permutation(n_cities) for _ in range(pop_size)])

In [3]:
# ── SCAFFOLD: Selection (TICKET-07) ─────────────────────────────

def tournament_select(population, fitnesses, k, rng):
    candidates = rng.choice(len(population), size=k, replace=False)
    return population[candidates[np.argmin(fitnesses[candidates])]].copy()


def roulette_select(population, fitnesses, rng):
    inv = 1.0 / fitnesses
    return population[rng.choice(len(population), p=inv / inv.sum())].copy()


# ── SCAFFOLD: Crossover (TICKET-08) ─────────────────────────────

def pmx(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    seg = set(child[pt1:pt2 + 1])
    for i in range(pt1, pt2 + 1):
        v = parent_b[i]
        if v not in seg:
            j = i
            while pt1 <= j <= pt2:
                j = np.where(parent_b == parent_a[j])[0][0]
            child[j] = v
    child[child == -1] = parent_b[child == -1]
    return child


def ox(parent_a, parent_b, pt1, pt2):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    seg = set(child[pt1:pt2 + 1])
    rem = [parent_b[(pt2 + 1 + i) % n] for i in range(n)
           if parent_b[(pt2 + 1 + i) % n] not in seg]
    j = 0
    for i in range(n):
        idx = (pt2 + 1 + i) % n
        if child[idx] == -1:
            child[idx] = rem[j]
            j += 1
    return child


def cx(parent_a, parent_b):
    n = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    visited = np.zeros(n, dtype=bool)
    pos_in_a = {v: i for i, v in enumerate(parent_a)}
    cycle = 0
    for start in range(n):
        if visited[start]:
            continue
        idxs, i = [], start
        while not visited[i]:
            visited[i] = True
            idxs.append(i)
            i = pos_in_a[parent_b[i]]
        src = parent_a if cycle % 2 == 0 else parent_b
        for idx in idxs:
            child[idx] = src[idx]
        cycle += 1
    return child, cycle


def naive_crossover(parent_a, parent_b, pt, _unused=None):
    # Single-point crossover that does NOT preserve the permutation property.
    # Used as the primary operator so the repair mechanism is load-bearing
    # in every generation, in line with the assignment's repair-centric focus.
    n = len(parent_a)
    child = np.empty(n, dtype=parent_a.dtype)
    child[:pt] = parent_a[:pt]
    child[pt:] = parent_b[pt:]
    return child


# ── SCAFFOLD: Mutation (TICKET-09) ──────────────────────────────

def swap_mutation(tour, rate, rng):
    t = tour.copy()
    if rng.random() < rate:
        i, j = rng.choice(len(t), size=2, replace=False)
        t[i], t[j] = t[j], t[i]
    return t


def inversion_mutation(tour, rate, rng):
    t = tour.copy()
    if rng.random() < rate:
        i, j = sorted(rng.choice(len(t), size=2, replace=False))
        t[i:j + 1] = t[i:j + 1][::-1]
    return t

In [4]:
# ── SCAFFOLD: Repair Mechanism (TICKET-12) ──────────────────────

def diagnose(chromosome, n_cities):
    seen, dups = set(), []
    for pos, city in enumerate(chromosome):
        if city in seen:
            dups.append(pos)
        else:
            seen.add(int(city))
    return dups, sorted(set(range(n_cities)) - seen)


def repair(chromosome, n_cities, dist_matrix=None, strategy="random", rng=None):
    tour = np.asarray(chromosome).copy()
    dups, missing = diagnose(tour, n_cities)
    if not dups:
        return tour
    if strategy == "random":
        if rng is None:
            rng = np.random.default_rng()
        replacements = list(rng.permutation(missing))
        for pos, city in zip(dups, replacements):
            tour[pos] = city
    elif strategy == "position":
        for pos, city in zip(dups, missing):
            tour[pos] = city
    elif strategy == "nearest":
        if dist_matrix is None:
            raise ValueError("strategy='nearest' requires dist_matrix")
        remaining = list(missing)
        for pos in dups:
            nearest = min(remaining, key=lambda c: dist_matrix[tour[pos - 1], c])
            tour[pos] = nearest
            remaining.remove(nearest)
    return tour

### Scaffold: Diversity Metric (TICKET-14)

Pairwise Hamming distance averaged across all pairs, normalised to [0, 1].
This scaffold is O(n²) over the population and sufficient for populations
up to ~200. Replace with an optimised or sampled version from notebook 14.

In [5]:
def pairwise_hamming(population):
    pop = np.asarray(population)
    n = len(pop)
    if n < 2:
        return 0.0
    total = sum(np.sum(pop[i] != pop[i + 1:]) for i in range(n - 1))
    return float(total / (n * (n - 1) / 2 * pop.shape[1]))

---
## Experiment Configuration

Extends the GA parameters from notebook 10 with repair and seed fields.
`config_hash` is computed from all fields **except** seed, so runs that
differ only in seed share a hash prefix.

In [6]:
@dataclass
class ExperimentConfig:
    pop_size: int = 100
    n_generations: int = 500
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    tournament_k: int = 3
    elitism_count: int = 2
    selection_method: str = "tournament"
    crossover_method: str = "naive"
    mutation_method: str = "swap"
    repair_enabled: bool = True
    repair_strategy: str = "random"
    seed: int = 42

---
## GA Loop with Full Logging

Same generational GA as notebook 10, extended to:
- compute population diversity every generation,
- optionally apply repair after crossover + mutation,
- return a per-generation `DataFrame` alongside the best tour.

In [7]:
def run_experiment(config, dist_matrix):
    n_cities = dist_matrix.shape[0]
    rng = np.random.default_rng(config.seed)

    xover_fn = {"pmx": pmx, "ox": ox, "cx": cx, "naive": naive_crossover}[config.crossover_method]
    mutate_fn = {"swap": swap_mutation, "inversion": inversion_mutation}[config.mutation_method]

    population = random_population(config.pop_size, n_cities, rng)
    fitnesses = np.array([tour_length(ind, dist_matrix) for ind in population])

    records = []

    for gen in range(config.n_generations):
        diversity = pairwise_hamming(population)
        records.append({
            "generation": gen,
            "best_fitness": float(fitnesses.min()),
            "mean_fitness": float(fitnesses.mean()),
            "diversity": float(diversity),
        })

        new_pop = []

        elite_idx = np.argsort(fitnesses)[:config.elitism_count]
        for idx in elite_idx:
            new_pop.append(population[idx].copy())

        while len(new_pop) < config.pop_size:
            if config.selection_method == "tournament":
                p1 = tournament_select(population, fitnesses, config.tournament_k, rng)
                p2 = tournament_select(population, fitnesses, config.tournament_k, rng)
            else:
                p1 = roulette_select(population, fitnesses, rng)
                p2 = roulette_select(population, fitnesses, rng)

            if rng.random() < config.crossover_rate:
                if config.crossover_method == "cx":
                    child, _ = xover_fn(p1, p2)
                else:
                    pts = sorted(rng.choice(n_cities, size=2, replace=False))
                    child = xover_fn(p1, p2, pts[0], pts[1])
            else:
                child = p1.copy()

            child = mutate_fn(child, config.mutation_rate, rng)

            if config.repair_enabled:
                child = repair(child, n_cities, dist_matrix=dist_matrix,
                               strategy=config.repair_strategy, rng=rng)

            new_pop.append(child)

        population = np.array(new_pop[:config.pop_size])
        fitnesses = np.array([tour_length(ind, dist_matrix) for ind in population])

    diversity = pairwise_hamming(population)
    records.append({
        "generation": config.n_generations,
        "best_fitness": float(fitnesses.min()),
        "mean_fitness": float(fitnesses.mean()),
        "diversity": float(diversity),
    })

    best_idx = np.argmin(fitnesses)
    return {
        "log": pd.DataFrame(records),
        "best_tour": population[best_idx],
        "best_fitness": float(fitnesses[best_idx]),
    }

---
## CSV I/O

One CSV per (configuration, seed). Config parameters are embedded in every
row so downstream notebooks can `pd.concat` all files without losing
metadata. Filename encodes the config hash and seed for human inspection.

In [8]:
RESULTS_DIR = Path("../results")


def config_hash(config):
    d = asdict(config)
    d.pop("seed")
    return hashlib.md5(json.dumps(d, sort_keys=True).encode()).hexdigest()[:8]


def result_path(config):
    return RESULTS_DIR / f"{config_hash(config)}_seed{config.seed:04d}.csv"


def save_run(config, result):
    path = result_path(config)
    path.parent.mkdir(parents=True, exist_ok=True)
    df = result["log"].copy()
    for k, v in asdict(config).items():
        df[k] = v
    df.to_csv(path, index=False)
    return path


def is_completed(config):
    return result_path(config).exists()

---
## Experiment Grid

`build_grid` takes a dict of `{parameter: [values]}`, a list of seeds,
and optional base defaults. It returns one `ExperimentConfig` per
(combination × seed).

In [9]:
def build_grid(param_grid, seeds, base=None):
    if base is None:
        base = {}
    keys = list(param_grid.keys())
    configs = []
    for combo in product(*param_grid.values()):
        params = dict(zip(keys, combo))
        for seed in seeds:
            configs.append(ExperimentConfig(**{**base, **params, "seed": seed}))
    return configs

---
## Experiment Runner

`run_grid` executes all pending configurations, skipping any whose result
CSV already exists (resumability). Set `n_workers > 1` to parallelise
with `multiprocessing` (uses `fork` context for notebook compatibility).

In [10]:
_DIST_MATRIX = None


def _init_pool(dist_matrix):
    global _DIST_MATRIX
    _DIST_MATRIX = dist_matrix


def _run_one(config):
    return run_experiment(config, _DIST_MATRIX)


def run_grid(configs, dist_matrix, n_workers=1):
    pending = [c for c in configs if not is_completed(c)]
    done = len(configs) - len(pending)
    print(f"Total: {len(configs)} | Completed: {done} | Pending: {len(pending)}")

    if not pending:
        print("Nothing to run \u2014 all results already exist.")
        return []

    results = []
    t0 = time.time()

    if n_workers <= 1:
        for i, config in enumerate(pending):
            result = run_experiment(config, dist_matrix)
            path = save_run(config, result)
            results.append(path)
            elapsed = time.time() - t0
            eta = elapsed / (i + 1) * (len(pending) - i - 1)
            print(f"  [{i + 1}/{len(pending)}] {path.name}  "
                  f"best={result['best_fitness']:.0f}  "
                  f"({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")
    else:
        ctx = mp.get_context("fork")
        with ctx.Pool(n_workers, initializer=_init_pool,
                       initargs=(dist_matrix,)) as pool:
            for i, result in enumerate(pool.imap(_run_one, pending)):
                path = save_run(pending[i], result)
                results.append(path)
                elapsed = time.time() - t0
                eta = elapsed / (i + 1) * (len(pending) - i - 1)
                print(f"  [{i + 1}/{len(pending)}] {path.name}  "
                      f"best={result['best_fitness']:.0f}  "
                      f"({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

    print(f"\nDone: {len(results)} runs in {time.time() - t0:.1f}s")
    return results

---
## Demo — Single Configuration

Quick smoke test: one run with small parameters to verify end-to-end
wiring (experiment → CSV → reload).

In [11]:
coords, dist_matrix = load_tsp(Path("../data/TSP-dataset/kroA100.tsp"))
n_cities = dist_matrix.shape[0]
print(f"Loaded kroA100: {n_cities} cities")

Loaded kroA100: 100 cities


In [12]:
OPTIMAL = 21282  # kroA100 known optimal (TSPLIB)

demo_config = ExperimentConfig(
    pop_size=50,
    n_generations=100,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_k=3,
    elitism_count=2,
    selection_method="tournament",
    crossover_method="naive",
    mutation_method="swap",
    repair_enabled=True,
    repair_strategy="random",
    seed=42,
)

t0 = time.time()
result = run_experiment(demo_config, dist_matrix)
elapsed = time.time() - t0

print(f"Best fitness : {result['best_fitness']:.2f}")
print(f"Known optimal: {OPTIMAL:,}")
print(f"Gap          : {(result['best_fitness'] - OPTIMAL) / OPTIMAL * 100:.1f}%")
print(f"Wall time    : {elapsed:.1f}s")
print(f"\nPer-generation log (first 5 rows):")
result["log"].head()

Best fitness : 89127.80
Known optimal: 21,282
Gap          : 318.8%
Wall time    : 0.2s

Per-generation log (first 5 rows):


,generation,best_fitness,mean_fitness,diversity
0,0,147937.568775,170274.610456,0.990261
1,1,147937.568775,164530.848996,0.961608
2,2,139904.966715,158621.458284,0.901298
3,3,136868.132624,155204.197506,0.882408
4,4,135630.593179,151910.975963,0.790367


In [13]:
path = save_run(demo_config, result)
print(f"Saved: {path}")
print(f"Size : {path.stat().st_size:,} bytes")

df = pd.read_csv(path)
print(f"Shape: {df.shape}")
print(f"Cols : {list(df.columns)}")
df.head()

Saved: ../results/649a28b7_seed0042.csv
Size : 11,878 bytes
Shape: (101, 16)
Cols : ['generation', 'best_fitness', 'mean_fitness', 'diversity', 'pop_size', 'n_generations', 'crossover_rate', 'mutation_rate', 'tournament_k', 'elitism_count', 'selection_method', 'crossover_method', 'mutation_method', 'repair_enabled', 'repair_strategy', 'seed']


,generation,best_fitness,mean_fitness,diversity,pop_size,n_generations,crossover_rate,mutation_rate,tournament_k,elitism_count,selection_method,crossover_method,mutation_method,repair_enabled,repair_strategy,seed
0,0,147937.568775,170274.610456,0.990261,50,100,0.8,0.2,3,2,tournament,naive,swap,True,random,42
1,1,147937.568775,164530.848996,0.961608,50,100,0.8,0.2,3,2,tournament,naive,swap,True,random,42
2,2,139904.966715,158621.458284,0.901298,50,100,0.8,0.2,3,2,tournament,naive,swap,True,random,42
3,3,136868.132624,155204.197506,0.882408,50,100,0.8,0.2,3,2,tournament,naive,swap,True,random,42
4,4,135630.593179,151910.975963,0.790367,50,100,0.8,0.2,3,2,tournament,naive,swap,True,random,42


## Demo — Experiment Grid

Build a small 2×1×3 grid (crossover × mutation × seed) with reduced
parameters. Verifies grid construction, execution, CSV output, and
resumability.

In [14]:
demo_seeds = list(range(1, 4))

demo_grid = build_grid(
    param_grid={
        "crossover_method": ["pmx", "ox"],
        "mutation_method": ["swap"],
    },
    seeds=demo_seeds,
    base={
        "pop_size": 30,
        "n_generations": 50,
        "mutation_rate": 0.2,
        "repair_enabled": True,
        "repair_strategy": "random",
    },
)

print(f"Grid: {len(demo_grid)} configurations")
for c in demo_grid:
    print(f"  {config_hash(c)} | xover={c.crossover_method} seed={c.seed}")

Grid: 6 configurations
  976d5e2d | xover=pmx seed=1
  976d5e2d | xover=pmx seed=2
  976d5e2d | xover=pmx seed=3
  0fc01629 | xover=ox seed=1
  0fc01629 | xover=ox seed=2
  0fc01629 | xover=ox seed=3


In [15]:
run_grid(demo_grid, dist_matrix, n_workers=1)

Total: 6 | Completed: 6 | Pending: 0
Nothing to run — all results already exist.


[]

In [16]:
print("Re-running the same grid (all should be skipped):")
run_grid(demo_grid, dist_matrix, n_workers=1)

Re-running the same grid (all should be skipped):
Total: 6 | Completed: 6 | Pending: 0
Nothing to run — all results already exist.


[]